<a href="https://colab.research.google.com/github/luanavarelaa/Puc-Campinas/blob/main/ACI_Projeto_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import glob
import numpy as np
from PIL import Image
from tqdm import tqdm
import kagglehub

In [ ]:
# Download latest version
path = kagglehub.dataset_download("mateuszbuda/lgg-mri-segmentation")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'lgg-mri-segmentation' dataset.
Path to dataset files: /kaggle/input/lgg-mri-segmentation


In [ ]:

# 1. Identificar o diretório base das pastas dos pacientes
base_dir = os.path.join(path, "kaggle_3m") if os.path.exists(os.path.join(path, "kaggle_3m")) else path

# 2. Criar a estrutura de diretórios para as classes binárias
output_dir = "dataset_lgg_processado"
dir_tumor = os.path.join(output_dir, "tumor")
dir_non_tumor = os.path.join(output_dir, "non_tumor")

os.makedirs(dir_tumor, exist_ok=True)
os.makedirs(dir_non_tumor, exist_ok=True)

# 3. Mapear todas as imagens das fatias, ignorando os arquivos que são máscaras
image_paths = glob.glob(f"{base_dir}/*/*[!_mask].tif")
print(f"Total de fatias axiais encontradas: {len(image_paths)}")

# 4. Iterar sobre os exames, verificar anotação tumoral e salvar
for img_path in tqdm(image_paths, desc="Processando e rotulando"):
    # Constrói o caminho esperado da máscara correspondente
    mask_path = img_path.replace(".tif", "_mask.tif")

    if not os.path.exists(mask_path):
        continue

    # Carrega a imagem anatômica e a máscara
    img = Image.open(img_path)
    mask = Image.open(mask_path)
    mask_arr = np.array(mask)

    # Critério: se houver algum pixel > 0, há lesão (Tumor); se for 0, é saudável
    has_tumor = np.max(mask_arr) > 0

    # Garante a dimensão 256x256 exigida pelo estudo
    if img.size != (256, 256):
        img = img.resize((256, 256))

    # Nome do arquivo de saída em PNG
    filename = os.path.basename(img_path).replace(".tif", ".png")

    # Salva na pasta correspondente
    if has_tumor:
        img.save(os.path.join(dir_tumor, filename))
    else:
        img.save(os.path.join(dir_non_tumor, filename))

# 5. Resumo da distribuição do dataset
n_tumor = len(os.listdir(dir_tumor))
n_non_tumor = len(os.listdir(dir_non_tumor))
total = n_tumor + n_non_tumor

print("\n--- Separação Concluída ---")
print(f"Fatias com Tumor (Classe 1): {n_tumor} ({(n_tumor/total)*100:.2f}%)")
print(f"Fatias Sem Tumor (Classe 0): {n_non_tumor} ({(n_non_tumor/total)*100:.2f}%)")
print(f"Total de fatias prontas: {total}")

Total de fatias axiais encontradas: 3929


Processando e rotulando: 100%|██████████| 3929/3929 [03:00<00:00, 21.77it/s]


--- Separação Concluída ---
Fatias com Tumor (Classe 1): 1373 (34.95%)
Fatias Sem Tumor (Classe 0): 2556 (65.05%)
Total de fatias prontas: 3929
